# Stacked U-Net Instance Segmentation（2 階段）

以 `unet_instance_segmentation.ipynb`（單階段）為基礎，**只改模型架構**，資料、增強、loss、post-processing、評估完全相同，方便直接比較。

```
image ──> Stage 1: U-Net (ResNet34) ──> out1 (14 ch: semantic 11 + center 1 + offset 2)
  │                                       │
  └──────── concat [image, softmax(sem1), center1, offset1] ──> Stage 2: U-Net (ResNet18) ──> Δ
                                                                         out2 = out1 + Δ
```

設計重點（參考 Stacked Hourglass）：
1. **第二階段的輸入 = 原圖 + 第一階段的預測**：讓它看到整張預測圖，修正不一致的地方（同一物體兩個中心、兩個物體只有一個中心、offset 方向亂掉）。
2. **殘差修正**：`out2 = out1 + Δ`，第二階段只學「哪裡要改」。最後一層初始化為 0，訓練一開始 `out2 == out1`。
3. **兩階段都算 loss（intermediate supervision）**：`loss = 0.5 × loss(out1) + 1.0 × loss(out2)`。
4. 推論 / 評估只用 `out2`；最後另外比較 `out1` vs `out2` 的 mAP，確認第二階段真的有幫助。

## 設定

In [ ]:
import os, json, math, datetime, contextlib, io, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

image_dir = "Dataset/img"
annotation_dir = "Dataset/annotations"
train_json_path = os.path.join(annotation_dir, "multiclass_train.json")
test_json_path = os.path.join(annotation_dir, "multiclass_test.json")

CFG = {
    # 資料
    "input_size": (512, 384),     # (W, H)，都要能被 32 整除；資料大多是 640x480（4:3）
    "center_sigma": 6,            # center heatmap 的 Gaussian sigma（input 解析度下的 pixel）
    "val_ratio": 0.1,
    "seed": 42,
    "use_rfs": True,              # Repeat Factor Sampling
    "rfs_threshold": 0.1,
    # 模型
    "encoder": "resnet34",
    "encoder_weights": "imagenet",
    "decoder_channels": (256, 128, 64, 64, 64),   # 最後一層加寬，給 offset 回歸多一點容量
    # stacked：第二階段
    "stage2_encoder": "resnet18",
    "stage2_encoder_weights": "imagenet",     # in_channels=17 時 smp 會自動改第一層，仍可用預訓練權重
    "stage1_loss_weight": 0.5,                # loss = w × loss(out1) + loss(out2)
    "stage1_init": None,                      # 例："best_unet_instance.pth"，用單階段權重初始化 stage 1（訓練較快，但就不是公平比較）
    # 訓練
    "batch_size": 8,              # 顯存比單階段多約 6 成，OOM 就改 6 或 4
    "num_epochs": 40,
    "lr_encoder": 1e-4,
    "lr_decoder": 5e-4,
    "weight_decay": 1e-4,
    "warmup_iters": 500,
    "grad_clip": 10.0,
    "amp": True,
    "eval_every": 2,              # 每幾個 epoch 算一次 val mAP（post-processing 比較慢）
    # loss 權重（Panoptic-DeepLab 原論文的設定）
    "w_sem": 1.0,
    "w_center": 200.0,
    "w_offset": 0.01,
    "offset_scale": 32.0,         # offset head 輸出 x 這個值 = pixel；head 初始輸出接近 0，不放大的話很難長到幾十 pixel
    # post-processing
    "center_thr": 0.1,
    "nms_kernel": 7,
    "top_k": 200,
    "min_area": 16,
    "max_dets": 100,
    "ckpt_path": "best_unet_instance_stacked.pth",
}

with open(train_json_path) as f:
    data = json.load(f)
with open(test_json_path) as f:
    test_data = json.load(f)

NUM_CLASSES = len(data["categories"]) + 1   # +1 是背景
id_to_name = {0: "background", **{c["id"]: c["name"] for c in data["categories"]}}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(CFG["seed"])
print(f"device: {device}  NUM_CLASSES: {NUM_CLASSES}")

## Dataset

每張圖的處理流程：
1. 把所有 polygon 解成 mask，畫進一張 **instance id map**（0 = 背景，1..K = 第幾個物件）。面積大的先畫、小的後畫，重疊時小物件不會被蓋掉。
2. image（bilinear）和 instance map（nearest）一起 resize 到 `input_size`。
3. 訓練時做水平翻轉 + ColorJitter。**翻轉是在 instance map 上做，之後才算 center / offset**，所以 offset 的 dx 方向自動正確。
4. 從 instance map 產生 semantic / center / offset 三種 target。

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def make_targets(inst_map, inst_labels, sigma):
    """inst_map: (H, W) int，0 = 背景、k = 第 k 個物件；inst_labels[k-1] 是它的類別"""
    H, W = inst_map.shape
    semantic = np.zeros((H, W), dtype=np.int64)
    center = np.zeros((H, W), dtype=np.float32)
    offset = np.zeros((2, H, W), dtype=np.float32)
    offset_weight = np.zeros((H, W), dtype=np.float32)

    r = int(3 * sigma)
    g = np.arange(-r, r + 1, dtype=np.float32)
    gaussian = np.exp(-(g[:, None] ** 2 + g[None, :] ** 2) / (2 * sigma ** 2))

    for k, label in enumerate(inst_labels, start=1):
        ys, xs = np.nonzero(inst_map == k)
        if len(ys) == 0:          # resize 後消失的小物件
            continue
        semantic[ys, xs] = label

        cy, cx = ys.mean(), xs.mean()          # 質心當中心
        offset[0, ys, xs] = cy - ys
        offset[1, ys, xs] = cx - xs
        offset_weight[ys, xs] = 1.0

        # 只在中心附近 (2r+1)^2 的窗口貼 Gaussian，比整張圖算快很多
        icy, icx = int(round(cy)), int(round(cx))
        y0, y1 = max(0, icy - r), min(H, icy + r + 1)
        x0, x1 = max(0, icx - r), min(W, icx + r + 1)
        patch = gaussian[y0 - (icy - r):y1 - (icy - r), x0 - (icx - r):x1 - (icx - r)]
        center[y0:y1, x0:x1] = np.maximum(center[y0:y1, x0:x1], patch)

    return {
        "semantic": torch.from_numpy(semantic),
        "center": torch.from_numpy(center)[None],
        "offset": torch.from_numpy(offset),
        "offset_weight": torch.from_numpy(offset_weight)[None],
    }


class UIIS10KUNetInstanceDataset(Dataset):
    def __init__(self, coco, image_dir, input_size, sigma, train=False, keep_empty=False):
        self.image_dir = image_dir
        self.input_size = input_size
        self.sigma = sigma
        self.train = train
        self.color_jitter = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02)

        self.anns_by_img = {}
        for ann in coco["annotations"]:
            self.anns_by_img.setdefault(ann["image_id"], []).append(ann)

        if keep_empty:
            self.images = list(coco["images"])
        else:
            self.images = [img for img in coco["images"] if img["id"] in self.anns_by_img]

    def __len__(self):
        return len(self.images)

    def load(self, idx):
        """回傳 resize 後的 PIL image、instance map、每個 instance 的類別（不做增強）"""
        info = self.images[idx]
        img = Image.open(os.path.join(self.image_dir, info["file_name"])).convert("RGB")
        width, height = img.size

        anns = sorted(self.anns_by_img.get(info["id"], []), key=lambda a: -a["area"])
        inst_map = np.zeros((height, width), dtype=np.int32)
        labels = []
        for ann in anns:
            rles = maskUtils.frPyObjects(ann["segmentation"], height, width)
            m = maskUtils.decode(rles)
            if m.ndim == 3:
                m = m.max(axis=2)
            labels.append(ann["category_id"])
            inst_map[m > 0] = len(labels)

        img = img.resize(self.input_size, Image.BILINEAR)
        inst_map = np.array(Image.fromarray(inst_map).resize(self.input_size, Image.NEAREST))
        return img, inst_map, np.array(labels, dtype=np.int64), info

    def __getitem__(self, idx):
        img, inst_map, labels, info = self.load(idx)

        if self.train:
            img = self.color_jitter(img)
            # 水平翻轉（用 torch 的 RNG，DataLoader 的每個 worker 會自動拿到不同 seed）
            if torch.rand(1).item() < 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                inst_map = inst_map[:, ::-1].copy()

        targets = make_targets(inst_map, labels, self.sigma)

        image = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD

        meta = {"image_id": info["id"], "orig_h": info["height"], "orig_w": info["width"]}
        return image, targets, meta


def denormalize(image):
    return (image.cpu() * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)

## 切 train / val，建立 DataLoader

切法跟 Mask R-CNN 版一致（只保留有標註的圖、seed 42、10%），val 集合完全相同，mAP 可以直接比較。
測試集 `keep_empty=True`：評估時一定要保留沒有標註的圖片。

In [ ]:
ds_kwargs = dict(input_size=CFG["input_size"], sigma=CFG["center_sigma"])
train_full = UIIS10KUNetInstanceDataset(data, image_dir, train=True, **ds_kwargs)
eval_full = UIIS10KUNetInstanceDataset(data, image_dir, train=False, **ds_kwargs)
assert [i["id"] for i in train_full.images] == [i["id"] for i in eval_full.images]

perm = np.random.default_rng(CFG["seed"]).permutation(len(train_full))
n_val = int(len(perm) * CFG["val_ratio"])
val_idx, train_idx = perm[:n_val], perm[n_val:]

test_ds = UIIS10KUNetInstanceDataset(test_data, image_dir, train=False, keep_empty=True, **ds_kwargs)

In [ ]:
train_ds = Subset(train_full, train_idx)
val_ds = Subset(eval_full, val_idx)
val_img_ids = [eval_full.images[i]["id"] for i in val_idx]
test_img_ids = [img["id"] for img in test_ds.images]

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

### Repeat Factor Sampling（類別不平衡）

跟 Mask R-CNN 版相同：類別 c 出現在 f_c 比例的圖片中，repeat factor = max(1, sqrt(t / f_c))；每張圖取它包含的類別中最大的 factor 當抽樣權重。

In [ ]:
def repeat_factors(dataset_full, indices, t):
    img_cats = []
    for i in indices:
        anns = dataset_full.anns_by_img.get(dataset_full.images[i]["id"], [])
        img_cats.append({a["category_id"] for a in anns})

    n = len(img_cats)
    freq = {}
    for cats in img_cats:
        for c in cats:
            freq[c] = freq.get(c, 0) + 1
    cat_rf = {c: max(1.0, math.sqrt(t / (k / n))) for c, k in freq.items()}
    img_rf = [max((cat_rf[c] for c in cats), default=1.0) for cats in img_cats]
    return img_rf, cat_rf


if CFG["use_rfs"]:
    img_rf, cat_rf = repeat_factors(train_full, train_idx, CFG["rfs_threshold"])
    for c in sorted(cat_rf):
        print(f"  {id_to_name[c]:>12s}: repeat factor {cat_rf[c]:.2f}")
    sampler = WeightedRandomSampler(img_rf, num_samples=len(train_ds), replacement=True)
else:
    sampler = None

loader_kwargs = dict(num_workers=4, pin_memory=True)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=(sampler is None),
                          sampler=sampler, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False, **loader_kwargs)

## 檢查 target：semantic / center heatmap / offset 有沒有對齊

In [ ]:
def offset_to_rgb(offset, weight):
    """offset 方向 → 色相，長度 → 亮度（背景是黑色）"""
    import matplotlib.colors as mcolors
    dy, dx = offset[0].numpy(), offset[1].numpy()
    ang = (np.arctan2(dy, dx) + np.pi) / (2 * np.pi)
    mag = np.sqrt(dy ** 2 + dx ** 2)
    mag = mag / (mag.max() + 1e-6)
    hsv = np.stack([ang, np.ones_like(ang), mag * weight[0].numpy()], axis=-1)
    return mcolors.hsv_to_rgb(hsv)


torch.manual_seed(0)
image, tgt, meta = train_ds[0]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].imshow(denormalize(image).permute(1, 2, 0)); axes[0].set_title("augmented image")
axes[1].imshow(tgt["semantic"], vmin=0, vmax=NUM_CLASSES - 1, cmap="tab20"); axes[1].set_title("semantic")
axes[2].imshow(tgt["center"][0], cmap="hot"); axes[2].set_title("center heatmap")
axes[3].imshow(offset_to_rgb(tgt["offset"], tgt["offset_weight"])); axes[3].set_title("offset (hue = direction)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 建立模型

- `build_model`：跟單階段版完全相同（`smp.Unet` + 3 個 head），當作 Stage 1。
- `StackedUNetInstance`：Stage 1 + Stage 2。Stage 2 的輸入是 `3 + 14 = 17` 個 channel，輸出同樣是 14 channel 的「修正量」。
- `forward(x, return_all=True)` 回傳 `(out1, out2)`，訓練和比較兩階段時用；預設只回傳 `out2`，所以後面的 `evaluate` / 視覺化不用改。

In [ ]:
def conv_head(in_ch, out_ch, mid_ch=64):
    return nn.Sequential(
        nn.Conv2d(in_ch, mid_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(mid_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(mid_ch, out_ch, kernel_size=1),
    )


class InstanceHeads(nn.Module):
    def __init__(self, in_ch, num_classes):
        super().__init__()
        self.semantic = conv_head(in_ch, num_classes)
        self.center = conv_head(in_ch, 1)
        self.offset = conv_head(in_ch, 2)

    def forward(self, x):
        return torch.cat([self.semantic(x), self.center(x), self.offset(x)], dim=1)


def build_model(num_classes):
    model = smp.Unet(
        encoder_name=CFG["encoder"],
        encoder_weights=CFG["encoder_weights"],
        in_channels=3,
        classes=num_classes,
        decoder_channels=CFG["decoder_channels"],
    )
    model.segmentation_head = InstanceHeads(CFG["decoder_channels"][-1], num_classes)
    return model


def split_outputs(output):
    return {
        "semantic": output[:, :NUM_CLASSES],
        "center": output[:, NUM_CLASSES:NUM_CLASSES + 1],
        "offset": output[:, NUM_CLASSES + 1:] * CFG["offset_scale"],
    }


class StackedUNetInstance(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.num_classes = num_classes
        out_ch = num_classes + 3
        self.stage1 = build_model(num_classes)

        self.stage2 = smp.Unet(
            encoder_name=CFG["stage2_encoder"],
            encoder_weights=CFG["stage2_encoder_weights"],
            in_channels=3 + out_ch,
            classes=num_classes,
            decoder_channels=CFG["decoder_channels"],
        )
        self.stage2.segmentation_head = InstanceHeads(CFG["decoder_channels"][-1], num_classes)

        # 最後一層 1x1 conv 歸零 → 一開始 Δ = 0、out2 == out1，第二階段從「不修改」開始學
        for head in [self.stage2.segmentation_head.semantic,
                     self.stage2.segmentation_head.center,
                     self.stage2.segmentation_head.offset]:
            nn.init.zeros_(head[-1].weight)
            nn.init.zeros_(head[-1].bias)

    def stage2_input(self, x, out1):
        C = self.num_classes
        sem_prob = F.softmax(out1[:, :C].float(), dim=1).to(out1.dtype)   # logits 數值範圍大，轉成機率比較好學
        return torch.cat([x, sem_prob, out1[:, C:]], dim=1)               # center、offset（÷32 前的值）直接用

    def forward(self, x, return_all=False):
        out1 = self.stage1(x)
        out2 = out1 + self.stage2(self.stage2_input(x, out1))
        return (out1, out2) if return_all else out2

    @property
    def encoders(self):
        return [self.stage1.encoder, self.stage2.encoder]


def build_stacked_model(num_classes):
    model = StackedUNetInstance(num_classes)
    if CFG["stage1_init"]:
        model.stage1.load_state_dict(torch.load(CFG["stage1_init"], map_location="cpu"))
        print("stage 1 initialized from", CFG["stage1_init"])
    n1 = sum(p.numel() for p in model.stage1.parameters()) / 1e6
    n2 = sum(p.numel() for p in model.stage2.parameters()) / 1e6
    print(f"params: stage1 {n1:.1f}M  stage2 {n2:.1f}M  total {n1 + n2:.1f}M")
    return model

## Loss

- **semantic**：CE + Dice（同 `sematic_segmentation.ipynb` 的 `CEDiceLoss`）
- **center**：整張圖的 MSE，權重 200（heatmap 大部分是 0，值很小，要放大）
- **offset**：只在物件像素上算 L1，權重 0.01（offset 單位是 pixel，數值很大）

In [ ]:
class CEDiceLoss(nn.Module):
    def __init__(self, num_classes, dice_weight=1.0, eps=1e-6, class_weights=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.num_classes = num_classes
        self.dice_weight = dice_weight
        self.eps = eps

    def forward(self, logits, target):
        ce = self.ce(logits, target)

        prob = F.softmax(logits, dim=1)
        target_onehot = F.one_hot(target, self.num_classes).permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        inter = (prob * target_onehot).sum(dims)
        union = prob.sum(dims) + target_onehot.sum(dims)
        dice_per_class = (2 * inter + self.eps) / (union + self.eps)

        present = target_onehot.sum(dims) > 0
        if present.any():
            dice_loss = 1 - dice_per_class[present].mean()
        else:
            dice_loss = 1 - dice_per_class.mean()

        return ce + self.dice_weight * dice_loss


class InstanceLoss(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.sem_loss = CEDiceLoss(num_classes)

    def forward(self, output, targets):
        out = split_outputs(output.float())   # AMP 下 loss 一律用 float32 算

        sem = self.sem_loss(out["semantic"], targets["semantic"])
        center = F.mse_loss(out["center"], targets["center"])

        w = targets["offset_weight"]
        offset = (F.l1_loss(out["offset"], targets["offset"], reduction="none") * w).sum() / (w.sum() * 2 + 1)

        total = CFG["w_sem"] * sem + CFG["w_center"] * center + CFG["w_offset"] * offset
        return total, {"sem": sem.item(), "center": center.item(), "offset": offset.item()}

## Post-processing：把三個輸出組成 instance

全部在 GPU 上、向量化完成（只有最後產生 mask 是逐物件）。
`postprocess` 回傳的 mask 是 `input_size` 解析度；`predict_coco_results` 會再 resize 回原圖大小、轉成 COCO RLE。

In [ ]:
@torch.no_grad()
def find_centers(center, thr, nms_kernel, top_k):
    """center: (H, W) → 峰值座標 (K, 2)[y, x] 和分數 (K,)"""
    pooled = F.max_pool2d(center[None, None], nms_kernel, stride=1, padding=nms_kernel // 2)[0, 0]
    keep = (center == pooled) & (center > thr)
    ys, xs = torch.nonzero(keep, as_tuple=True)
    scores = center[ys, xs]
    if len(scores) > top_k:
        idx = scores.topk(top_k).indices
        ys, xs, scores = ys[idx], xs[idx], scores[idx]
    return torch.stack([ys, xs], dim=1).float(), scores


@torch.no_grad()
def postprocess(sem_prob, center, offset, cfg=CFG):
    """
    sem_prob: (C, H, W) softmax 後；center: (H, W)；offset: (2, H, W)
    回傳 masks (K, H, W) bool、labels (K,)、scores (K,)
    """
    C, H, W = sem_prob.shape
    empty = (torch.zeros((0, H, W), dtype=torch.bool, device=sem_prob.device),
             torch.zeros((0,), dtype=torch.long), torch.zeros((0,)))

    sem_pred = sem_prob.argmax(0)
    fg = sem_pred > 0
    centers, center_scores = find_centers(center, cfg["center_thr"], cfg["nms_kernel"], cfg["top_k"])
    if len(centers) == 0 or not fg.any():
        return empty

    # 1) 每個前景像素：位置 + offset → 最近的中心
    ys, xs = torch.nonzero(fg, as_tuple=True)
    voted = torch.stack([ys + offset[0, ys, xs], xs + offset[1, ys, xs]], dim=1)
    assign = torch.cdist(voted, centers).argmin(dim=1)          # (N_fg,)
    K = len(centers)

    # 2) 每個物件的類別 = semantic 多數決（只在前景類別中投票）
    pix_cls = sem_pred[ys, xs]
    votes = torch.bincount(assign * C + pix_cls, minlength=K * C).view(K, C)
    votes[:, 0] = 0
    inst_cls = votes.argmax(dim=1)                               # (K,)
    area = votes.sum(dim=1)

    # 3) 分數 = 中心 heatmap 值 × 物件內「該類別」的平均機率
    pix_prob = sem_prob[inst_cls[assign], ys, xs]
    mean_prob = torch.bincount(assign, weights=pix_prob, minlength=K) / area.clamp(min=1)
    inst_scores = center_scores.clamp(max=1) * mean_prob

    valid = area >= cfg["min_area"]
    if not valid.any():
        return empty
    order = inst_scores[valid].argsort(descending=True)[:cfg["max_dets"]]
    keep_ids = torch.nonzero(valid).squeeze(1)[order]

    inst_map = torch.full((H, W), -1, dtype=torch.long, device=sem_prob.device)
    inst_map[ys, xs] = assign
    masks = inst_map[None] == keep_ids[:, None, None]
    return masks, inst_cls[keep_ids].cpu(), inst_scores[keep_ids].cpu()


def model_forward(model, images, **kwargs):
    with torch.autocast(device_type="cuda", dtype=torch.float16,
                        enabled=CFG["amp"] and device.type == "cuda"):
        return model(images, **kwargs)

## 訓練 / 評估函式

- `train_one_epoch`：AdamW + AMP，scheduler 每個 iteration 呼叫一次（warmup → cosine）；loss = 0.5 × stage 1 + stage 2
- `stacked_loss`：兩階段 loss 加權相加，回傳的分項（sem / center / offset）是 stage 2 的
- `evaluate(..., stage=1 或 2)`：可以指定用哪一階段的輸出做預測，用來比較兩階段
- `evaluate`：一次跑完，同時算 val loss 和 COCO 預測結果（不用跑兩遍）
- `coco_eval` / `per_class_ap`：跟 Mask R-CNN 版一樣

In [ ]:
def build_optimizer_and_scheduler(model, iters_per_epoch):
    encoder_params = [p for enc in model.encoders for p in enc.parameters()]
    encoder_ids = {id(p) for p in encoder_params}
    other_params = [p for p in model.parameters() if id(p) not in encoder_ids]

    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": CFG["lr_encoder"]},
        {"params": other_params, "lr": CFG["lr_decoder"]},
    ], weight_decay=CFG["weight_decay"])

    total = CFG["num_epochs"] * iters_per_epoch
    warmup = CFG["warmup_iters"]

    def lr_lambda(it):   # 以 iteration 為單位：線性 warmup，之後 cosine 衰減
        if it < warmup:
            return 0.001 + (1 - 0.001) * it / warmup
        progress = (it - warmup) / max(1, total - warmup)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def to_device(targets):
    return {k: v.to(device, non_blocking=True) for k, v in targets.items()}


def stacked_loss(out1, out2, targets):
    loss1, parts1 = criterion(out1, targets)
    loss2, parts2 = criterion(out2, targets)
    parts2["stage1_total"] = loss1.item()
    return CFG["stage1_loss_weight"] * loss1 + loss2, parts2


def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss, parts, n = 0.0, {"sem": 0.0, "center": 0.0, "offset": 0.0, "stage1_total": 0.0}, 0

    for images, targets, _ in loader:
        images = images.to(device, non_blocking=True)
        targets = to_device(targets)

        out1, out2 = model_forward(model, images, return_all=True)
        loss, loss_parts = stacked_loss(out1, out2, targets)

        if not torch.isfinite(loss):
            print("  !! non-finite loss, skip batch:", loss_parts)
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            continue

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        for k in parts:
            parts[k] += loss_parts[k]
        n += 1

    n = max(n, 1)
    return total_loss / n, {k: v / n for k, v in parts.items()}


def encode_instances(masks, labels, scores, img_id, orig_h, orig_w):
    """input_size 的 mask → 原圖大小 → COCO RLE"""
    results = []
    if len(masks) == 0:
        return results
    masks = F.interpolate(masks[None].float(), size=(orig_h, orig_w), mode="bilinear",
                          align_corners=False)[0] > 0.5
    masks = masks.cpu().numpy().astype(np.uint8)
    for m, lab, sc in zip(masks, labels.tolist(), scores.tolist()):
        if m.sum() == 0:
            continue
        rle = maskUtils.encode(np.asfortranarray(m))
        rle["counts"] = rle["counts"].decode("ascii")
        x, y, w, h = maskUtils.toBbox(rle).tolist()
        results.append({"image_id": img_id, "category_id": int(lab), "score": float(sc),
                        "segmentation": rle, "bbox": [x, y, w, h]})
    return results


@torch.no_grad()
def evaluate(model, loader, cfg=CFG, compute_loss=True, stage=2):
    model.eval()
    total_loss, n = 0.0, 0
    results = []

    for images, targets, meta in loader:
        images = images.to(device, non_blocking=True)
        out1, out2 = model_forward(model, images, return_all=True)
        output = (out1 if stage == 1 else out2).float()

        if compute_loss:
            loss, _ = criterion(output, to_device(targets))
            total_loss += loss.item()
            n += 1

        out = split_outputs(output)
        sem_prob = F.softmax(out["semantic"], dim=1)
        for b in range(images.shape[0]):
            masks, labels, scores = postprocess(sem_prob[b], out["center"][b, 0], out["offset"][b], cfg)
            results += encode_instances(masks, labels, scores, int(meta["image_id"][b]),
                                        int(meta["orig_h"][b]), int(meta["orig_w"][b]))

    return (total_loss / max(n, 1) if compute_loss else None), results


def coco_eval(coco_gt, results, img_ids, iou_type="segm", verbose=True):
    if len(results) == 0:
        print("no detections")
        return None
    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(results)
    E = COCOeval(coco_gt, coco_dt, iouType=iou_type)
    E.params.imgIds = list(img_ids)
    if verbose:
        E.evaluate(); E.accumulate(); E.summarize()
    else:
        with contextlib.redirect_stdout(io.StringIO()):
            E.evaluate(); E.accumulate(); E.summarize()
    return E


def per_class_ap(E):
    # precision: [IoU thresholds, recall, class, area range, max dets]
    precision = E.eval["precision"]
    out = {}
    for k, cat_id in enumerate(E.params.catIds):
        p = precision[:, :, k, 0, -1]
        p = p[p > -1]
        out[id_to_name[cat_id]] = float(p.mean()) if p.size else float("nan")
    return out


with contextlib.redirect_stdout(io.StringIO()):
    coco_train_gt = COCO(train_json_path)   # val 是從 train json 切出來的，GT 要用這個
    coco_test_gt = COCO(test_json_path)

criterion = InstanceLoss(NUM_CLASSES)

## 健康檢查：單一 batch 過擬合測試

確認資料流 + 三個 loss 都能正常下降，用獨立的 `sanity_model`，不影響正式訓練。

In [ ]:
sanity_model = build_stacked_model(NUM_CLASSES).to(device)
sanity_opt = torch.optim.AdamW(sanity_model.parameters(), lr=1e-3)
sanity_scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"] and device.type == "cuda")

images, targets, _ = next(iter(DataLoader(val_ds, batch_size=2)))
images, targets = images.to(device), to_device(targets)

sanity_model.train()
for step in range(60):
    out1, out2 = model_forward(sanity_model, images, return_all=True)
    loss, parts = stacked_loss(out1, out2, targets)
    sanity_opt.zero_grad(set_to_none=True)
    sanity_scaler.scale(loss).backward()
    sanity_scaler.step(sanity_opt)
    sanity_scaler.update()
    if step % 10 == 0:
        print(step, round(loss.item(), 4), {k: round(v, 4) for k, v in parts.items()})

del sanity_model, sanity_opt, images, targets
if device.type == "cuda":
    torch.cuda.empty_cache()

## 正式訓練

每個 epoch 算 train loss，每 `eval_every` 個 epoch 算 val loss + **val mask mAP**，用 mask mAP 挑最佳 checkpoint（存到 `best_unet_instance_stacked.pth`，不覆蓋單階段的 `best_unet_instance.pth`）。val loss / mAP 都是用 stage 2 的輸出算的。。

In [ ]:
model = build_stacked_model(NUM_CLASSES).to(device)
optimizer, scheduler = build_optimizer_and_scheduler(model, len(train_loader))
scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"] and device.type == "cuda")

history = {"epoch": [], "train_loss": [], "train_parts": [], "val_epoch": [],
           "val_loss": [], "val_segm_map": [], "val_segm_map50": [], "lr": []}
best_map, best_epoch = -1.0, -1

for epoch in range(CFG["num_epochs"]):
    t0 = time.time()
    lr_now = optimizer.param_groups[1]["lr"]
    train_loss, parts = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_parts"].append(parts)
    history["lr"].append(lr_now)

    msg = (f"Epoch [{epoch+1}/{CFG['num_epochs']}] lr {lr_now:.6f} | train loss {train_loss:.4f} "
           f"(stage2: sem {parts['sem']:.3f} ctr {parts['center']:.5f} off {parts['offset']:.2f} | stage1 {parts['stage1_total']:.3f})")

    if (epoch + 1) % CFG["eval_every"] == 0 or epoch + 1 == CFG["num_epochs"]:
        val_loss, results = evaluate(model, val_loader)
        E = coco_eval(coco_train_gt, results, val_img_ids, "segm", verbose=False)
        segm_map = E.stats[0] if E else 0.0
        segm_map50 = E.stats[1] if E else 0.0

        history["val_epoch"].append(epoch + 1)
        history["val_loss"].append(val_loss)
        history["val_segm_map"].append(segm_map)
        history["val_segm_map50"].append(segm_map50)
        msg += f" | val loss {val_loss:.4f} | val mask mAP {segm_map:.4f} (AP50 {segm_map50:.4f})"

        if segm_map > best_map:
            best_map, best_epoch = segm_map, epoch + 1
            torch.save(model.state_dict(), CFG["ckpt_path"])
            msg += "  -> saved best"

    print(msg + f"  [{time.time() - t0:.0f}s]")

torch.save(model.state_dict(), CFG["ckpt_path"].replace(".pth", "_last.pth"))
print(f"best val mask mAP {best_map:.4f} @ epoch {best_epoch}")

## 畫曲線並存檔

In [ ]:
def new_run_dir(run_name=None):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    name = f"{timestamp}_{run_name}" if run_name else timestamp
    run_dir = os.path.join("Result", name)
    os.makedirs(run_dir, exist_ok=True)
    return run_dir


run_dir = new_run_dir("unet_instance_stacked")
print("本次結果存放於:", run_dir)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(history["epoch"], history["train_loss"], label="train loss")
axes[0].plot(history["val_epoch"], history["val_loss"], "o-", label="val loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Total loss"); axes[0].legend()

for k, w in [("sem", CFG["w_sem"]), ("center", CFG["w_center"]), ("offset", CFG["w_offset"])]:
    axes[1].plot(history["epoch"], [p[k] * w for p in history["train_parts"]], label=f"{k} (x{w:g})")
axes[1].plot(history["epoch"], [p["stage1_total"] for p in history["train_parts"]], "--", label="stage1 total")
axes[1].set_xlabel("Epoch"); axes[1].set_title("Train loss parts (stage 2, weighted)"); axes[1].legend()

axes[2].plot(history["val_epoch"], history["val_segm_map"], "o-", label="mask mAP@[.5:.95]")
axes[2].plot(history["val_epoch"], history["val_segm_map50"], "o-", label="mask AP50")
axes[2].set_xlabel("Epoch"); axes[2].set_title("Validation mAP"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(run_dir, "curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## (選用) 在 val 上調 post-processing 參數

不用重新訓練，只換 `center_thr` / `nms_kernel` 重跑推論。挑到最好的再更新 `CFG`，之後測試集就會用新參數。

In [ ]:
model.load_state_dict(torch.load(CFG["ckpt_path"], map_location=device))

for thr in [0.05, 0.1, 0.2]:
    for k in [5, 7, 11]:
        cfg = {**CFG, "center_thr": thr, "nms_kernel": k}
        _, results = evaluate(model, val_loader, cfg=cfg, compute_loss=False)
        E = coco_eval(coco_train_gt, results, val_img_ids, "segm", verbose=False)
        print(f"center_thr {thr:<5} nms_kernel {k:<3} -> val mask mAP {E.stats[0] if E else 0:.4f}")

# 例：CFG.update(center_thr=0.1, nms_kernel=7)

## 測試集評估

載入最佳權重，在測試集（包含沒有標註的圖片）上算完整的 COCO 指標，以及每個類別的 mask AP。

In [ ]:
model.load_state_dict(torch.load(CFG["ckpt_path"], map_location=device))

_, test_results = evaluate(model, test_loader, compute_loss=False)

print("===== Test: segm =====")
E_test_segm = coco_eval(coco_test_gt, test_results, test_img_ids, "segm")

test_class_ap = per_class_ap(E_test_segm)
print("\n===== Test: per-class mask AP =====")
for name, ap in sorted(test_class_ap.items(), key=lambda x: -x[1]):
    print(f"  {name:>12s}: {ap:.4f}")

STAT_NAMES = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
              "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]

with open(os.path.join(run_dir, "history.json"), "w") as f:
    json.dump({
        "history": history,
        "config": CFG,
        "best_epoch": best_epoch,
        "best_val_segm_map": best_map,
        "test_segm": dict(zip(STAT_NAMES, map(float, E_test_segm.stats))),
        "test_per_class_mask_ap": test_class_ap,
    }, f, indent=2)

## 比較 Stage 1 vs Stage 2

同一個模型，分別用 `out1` 和 `out2` 做預測：
- `out2` 明顯比 `out1` 好 → 第二階段的修正有效
- 差不多 → 第二階段沒學到東西，只是多花算力

也請跟單階段版（`Result/*_unet_instance/history.json` 的 `test_segm`）比較：stage 1 在這裡有 intermediate supervision，本身也可能跟單階段版不同。

In [ ]:
STAGE_STATS = {}
for split, loader, gt, ids in [("val", val_loader, coco_train_gt, val_img_ids),
                               ("test", test_loader, coco_test_gt, test_img_ids)]:
    for stage in (1, 2):
        _, results = evaluate(model, loader, compute_loss=False, stage=stage)
        E = coco_eval(gt, results, ids, "segm", verbose=False)
        STAGE_STATS[f"{split}_stage{stage}"] = dict(zip(STAT_NAMES, map(float, E.stats))) if E else None
        if E:
            print(f"{split:>4s} stage {stage}: mask AP {E.stats[0]:.4f}  AP50 {E.stats[1]:.4f}  "
                  f"AP75 {E.stats[2]:.4f}  AP_small {E.stats[3]:.4f}  AR100 {E.stats[8]:.4f}")

with open(os.path.join(run_dir, "history.json")) as f:
    payload = json.load(f)
payload["stage_comparison"] = STAGE_STATS
with open(os.path.join(run_dir, "history.json"), "w") as f:
    json.dump(payload, f, indent=2)

## 視覺化：GT / 預測 instance / center heatmap / semantic

In [ ]:
def draw_instances(ax, image, masks, labels, scores=None, title=""):
    """image: (H, W, 3) float；masks: (K, H, W) bool"""
    ax.imshow(image)
    cmap = plt.get_cmap("tab20")
    overlay = np.zeros((*image.shape[:2], 4))
    for k, (m, lab) in enumerate(zip(masks, labels)):
        color = cmap(k % 20)
        overlay[m] = (*color[:3], 0.5)
        ys, xs = np.nonzero(m)
        if len(ys) == 0:
            continue
        text = id_to_name.get(int(lab), str(int(lab)))
        if scores is not None:
            text += f" {float(scores[k]):.2f}"
        ax.text(xs.min(), max(ys.min() - 3, 0), text, color=color, fontsize=8, weight="bold")
    ax.imshow(overlay)
    ax.set_title(title)
    ax.axis("off")


@torch.no_grad()
def show_gt_vs_pred(dataset_full, indices, n=3, score_threshold=0.3, seed=0, save_path=None):
    model.eval()
    idxs = np.random.default_rng(seed).choice(indices, n, replace=False)
    fig, axes = plt.subplots(n, 4, figsize=(24, 4.5 * n))
    axes = np.atleast_2d(axes)

    for row, i in zip(axes, idxs):
        img, inst_map, gt_labels, _ = dataset_full.load(i)
        img_np = np.array(img) / 255.0
        gt_masks = np.stack([inst_map == k for k in range(1, len(gt_labels) + 1)]) if len(gt_labels) else np.zeros((0, *inst_map.shape), bool)
        draw_instances(row[0], img_np, gt_masks, gt_labels, title=f"GT: {len(gt_labels)} object(s)")

        image, _, _ = dataset_full[i]
        out = split_outputs(model_forward(model, image[None].to(device)).float())
        sem_prob = F.softmax(out["semantic"], dim=1)[0]
        masks, labels, scores = postprocess(sem_prob, out["center"][0, 0], out["offset"][0])
        keep = scores > score_threshold
        draw_instances(row[1], img_np, masks[keep.to(masks.device)].cpu().numpy(), labels[keep], scores[keep],
                       title=f"Pred: {int(keep.sum())} object(s)")

        row[2].imshow(img_np)
        row[2].imshow(out["center"][0, 0].cpu().clamp(0, 1), cmap="hot", alpha=0.6)
        row[2].set_title("pred center heatmap"); row[2].axis("off")

        row[3].imshow(sem_prob.argmax(0).cpu(), vmin=0, vmax=NUM_CLASSES - 1, cmap="tab20")
        row[3].set_title("pred semantic"); row[3].axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()


show_gt_vs_pred(eval_full, val_idx, n=3, save_path=os.path.join(run_dir, "val_predictions.png"))
show_gt_vs_pred(test_ds, np.arange(len(test_ds)), n=3, save_path=os.path.join(run_dir, "test_predictions.png"))